# 08B — Physics-Safe AIA AlexNet-v2 Fold-2015 Largecap Benchmark

This notebook replaces the earlier exploratory AlexNet-v2 run with a **physics-safe** version.

## Scientific purpose

The aim is not to chase a metric through questionable computer-vision tricks. The aim is to test whether AlexNet can improve under a cleaner optimisation protocol while preserving the physical meaning of the AIA observations.

## Physics-safe rules used here

- Same chronological fold-2015 protocol:
  - train: 2010–2013
  - validation: 2014
  - test: 2015
- Same target: `label_48h_final` from the manifest/sample CSV only.
- No embedded NPZ label is used.
- No random horizontal flip.
- No random vertical flip.
- No random rotation.
- No random crop.
- Deterministic resize only, used as a computational baseline.
- Channel order is preserved.
- Class imbalance is handled with `pos_weight` only.
- No weighted random sampler.
- Threshold is selected by maximum TSS on validation and applied unchanged to test.
- Test data is never used for model or threshold selection.

## Normalisation policy

This notebook avoids per-image percentile normalisation because it can erase real brightness/intensity differences between active regions.

Instead, it computes **train-derived robust channel statistics** from the training split only and applies the same transformation to train, validation, and test. This is more defensible for forecasting because validation/test information is not used in preprocessing.


In [ ]:
from pathlib import Path
import json, csv, hashlib, math, os, random, subprocess, time
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

try:
    from IPython.display import display
except Exception:
    display = print

def repo_root() -> Path:
    try:
        return Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
    except Exception:
        return Path.cwd()

ROOT = repo_root()
print("Repo root:", ROOT)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)


In [ ]:
# -----------------------------
# Experiment configuration
# -----------------------------

@dataclass
class Config:
    experiment_name: str = "aia_alexnet_v2_physics_safe_fold2015_largecap_benchmark"
    run_mode: str = "largecap_v2_physics_safe"
    fold_id: str = "test_2015"
    label_col: str = "label_48h_final"

    # Reuse the exact 06B largecap sample files.
    train_samples: str = "results/metrics/aia_alexnet_fold2015_largecap_benchmark_largecap_train_samples.csv"
    val_samples: str = "results/metrics/aia_alexnet_fold2015_largecap_benchmark_largecap_val_samples.csv"
    test_samples: str = "results/metrics/aia_alexnet_fold2015_largecap_benchmark_largecap_test_samples.csv"

    cache_dir: str = "cache/gcs_npz_alexnet_fold2015"

    image_size: int = 224
    in_channels: int = 6
    batch_size: int = 32
    num_workers: int = 2

    epochs: int = 6
    learning_rate: float = 1e-4
    weight_decay: float = 5e-4
    dropout: float = 0.50

    # Physics-safe setting: no weighted sampler, no random spatial augmentation.
    use_weighted_sampler: bool = False
    spatial_augmentation: bool = False

    # Robust train-derived channel stats.
    stats_max_images: int = 1500
    stats_pixels_per_channel_per_image: int = 512

    threshold_grid_step: float = 0.0025
    seed: int = 42

CFG = Config()
seed_everything(CFG.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(CFG)
print("Device:", device)
assert CFG.use_weighted_sampler is False
assert CFG.spatial_augmentation is False


In [ ]:
# -----------------------------
# Load sample files
# -----------------------------

def read_samples(path: Path, split: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["split"] = split

    if "gcp_path" not in df.columns:
        raise ValueError(f"{path} must contain a gcp_path column. Columns: {df.columns.tolist()}")

    # Physics/causality rule: use manifest label only.
    if CFG.label_col not in df.columns:
        raise ValueError(
            f"{path} has no required manifest label column {CFG.label_col}. "
            f"Columns: {df.columns.tolist()}"
        )

    df[CFG.label_col] = df[CFG.label_col].astype(int)
    return df

train_df = read_samples(ROOT / CFG.train_samples, "train")
val_df = read_samples(ROOT / CFG.val_samples, "val")
test_df = read_samples(ROOT / CFG.test_samples, "test")

def summarise_split(df, name):
    pos = int(df[CFG.label_col].sum())
    rows = int(len(df))
    neg = rows - pos
    years = sorted(df["year"].dropna().unique().tolist()) if "year" in df.columns else []
    return {
        "split": name,
        "rows": rows,
        "positives": pos,
        "negatives": neg,
        "positive_rate": pos / rows if rows else float("nan"),
        "years": str(years),
    }

data_summary = [
    summarise_split(train_df, "train"),
    summarise_split(val_df, "val"),
    summarise_split(test_df, "test"),
]
data_summary_df = pd.DataFrame(data_summary)
display(data_summary_df)


In [ ]:
# -----------------------------
# Cache verification
# -----------------------------

CACHE = ROOT / CFG.cache_dir
CACHE.mkdir(parents=True, exist_ok=True)

def local_path_for_gcs(gcp_path: str) -> Path:
    base = Path(gcp_path).name
    h = hashlib.md5(gcp_path.encode("utf-8")).hexdigest()
    return CACHE / f"{h}_{base}"

required_paths = sorted(set(pd.concat([train_df, val_df, test_df])["gcp_path"].tolist()))
missing = [p for p in required_paths if not local_path_for_gcs(p).exists()]

print("Required unique NPZ files:", len(required_paths))
print("Missing cached files:", len(missing))
if missing[:5]:
    print("First missing:", missing[:5])

assert len(missing) == 0, "Cache is incomplete. Run scripts/prefetch_06b_missing.py before this notebook."


In [ ]:
# -----------------------------
# NPZ loading helpers
# -----------------------------

def pick_npz_array(npz) -> np.ndarray:
    preferred = ["x", "X", "image", "images", "data", "arr_0"]
    keys = list(npz.files)
    for k in preferred:
        if k in keys:
            arr = npz[k]
            if isinstance(arr, np.ndarray) and arr.ndim >= 2:
                return arr
    for k in keys:
        arr = npz[k]
        if isinstance(arr, np.ndarray) and arr.ndim >= 2:
            return arr
    raise ValueError(f"No image-like array found in NPZ keys={keys}")

def to_chw_six(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = np.squeeze(arr)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D AIA array, got shape={arr.shape}")
    if arr.shape[0] == CFG.in_channels:
        chw = arr
    elif arr.shape[-1] == CFG.in_channels:
        chw = np.transpose(arr, (2, 0, 1))
    else:
        raise ValueError(f"Cannot infer six-channel layout from shape={arr.shape}")
    return chw.astype(np.float32)

def load_raw_chw(gcp_path: str) -> np.ndarray:
    path = local_path_for_gcs(gcp_path)
    with np.load(path, allow_pickle=False) as npz:
        arr = pick_npz_array(npz)
    chw = to_chw_six(arr)
    chw = np.nan_to_num(chw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return chw


In [ ]:
# -----------------------------
# Train-derived robust channel statistics
# -----------------------------

METRICS_DIR = ROOT / "results/metrics"
MODELS_DIR = ROOT / "results/models"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

prefix = f"{CFG.experiment_name}_{CFG.run_mode}"
stats_path = METRICS_DIR / f"{prefix}_train_channel_stats.json"
progress_path = METRICS_DIR / f"{prefix}_progress.log"

def log_msg(msg: str):
    print(msg, flush=True)
    with open(progress_path, "a") as fh:
        fh.write(msg + "\n")

def deterministic_pixel_sample(channel_2d: np.ndarray, n: int, seed_key: str) -> np.ndarray:
    flat = channel_2d.reshape(-1)
    if flat.size <= n:
        return flat.astype(np.float32)
    # Stable deterministic seed from source identity.
    h = int(hashlib.md5(seed_key.encode("utf-8")).hexdigest()[:8], 16)
    rng = np.random.default_rng(h)
    idx = rng.choice(flat.size, size=n, replace=False)
    return flat[idx].astype(np.float32)

def compute_train_channel_stats() -> Dict:
    if stats_path.exists():
        log_msg(f"Loading existing train-derived channel stats: {stats_path}")
        return json.loads(stats_path.read_text())

    log_msg("Computing train-derived robust channel statistics...")
    train_paths = train_df["gcp_path"].drop_duplicates().tolist()
    if CFG.stats_max_images and len(train_paths) > CFG.stats_max_images:
        train_paths = train_paths[:CFG.stats_max_images]

    samples_by_channel = [[] for _ in range(CFG.in_channels)]
    t0 = time.time()

    for i, gcp_path in enumerate(train_paths, 1):
        chw = load_raw_chw(gcp_path)
        if chw.shape[0] != CFG.in_channels:
            raise ValueError(f"Expected {CFG.in_channels} channels, got {chw.shape} for {gcp_path}")

        for c in range(CFG.in_channels):
            vals = deterministic_pixel_sample(
                chw[c],
                CFG.stats_pixels_per_channel_per_image,
                seed_key=f"{gcp_path}|channel={c}|physics_safe_stats",
            )
            samples_by_channel[c].append(vals)

        if i % 100 == 0 or i == len(train_paths):
            log_msg(f"stats progress: {i}/{len(train_paths)} images elapsed_min={(time.time()-t0)/60:.1f}")

    channel_stats = []
    for c in range(CFG.in_channels):
        vals = np.concatenate(samples_by_channel[c]).astype(np.float32)
        vals = vals[np.isfinite(vals)]
        q01, q25, q50, q75, q99 = np.percentile(vals, [1, 25, 50, 75, 99])
        iqr = float(q75 - q25)
        if not np.isfinite(iqr) or iqr <= 1e-6:
            iqr = float(np.std(vals) + 1e-6)
        channel_stats.append({
            "channel_index": c,
            "q01": float(q01),
            "q25": float(q25),
            "median": float(q50),
            "q75": float(q75),
            "q99": float(q99),
            "iqr_or_std": float(iqr),
            "sample_count": int(vals.size),
        })

    payload = {
        "normalisation": "train_derived_channel_robust_q01_q99_clip_median_iqr_scale",
        "source_split": "train_only",
        "stats_max_images": CFG.stats_max_images,
        "stats_pixels_per_channel_per_image": CFG.stats_pixels_per_channel_per_image,
        "num_train_paths_used": len(train_paths),
        "channel_stats": channel_stats,
        "physics_note": (
            "Statistics are estimated from training images only and then applied unchanged to "
            "train/validation/test. No validation or test information is used."
        ),
    }
    stats_path.write_text(json.dumps(payload, indent=2))
    log_msg(f"Saved train-derived channel stats: {stats_path}")
    return payload

channel_stats_payload = compute_train_channel_stats()
channel_stats_payload


In [ ]:
# -----------------------------
# Physics-safe dataset
# -----------------------------

stats = channel_stats_payload["channel_stats"]
clip_lo = np.array([s["q01"] for s in stats], dtype=np.float32)[:, None, None]
clip_hi = np.array([s["q99"] for s in stats], dtype=np.float32)[:, None, None]
median = np.array([s["median"] for s in stats], dtype=np.float32)[:, None, None]
scale = np.array([s["iqr_or_std"] for s in stats], dtype=np.float32)[:, None, None]
scale = np.where(np.abs(scale) < 1e-6, 1.0, scale).astype(np.float32)

def physics_safe_normalise(chw: np.ndarray) -> np.ndarray:
    # Same train-derived transformation for all splits.
    x = np.clip(chw.astype(np.float32), clip_lo, clip_hi)
    x = (x - median) / scale
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return x

class AIANPZPhysicsSafeDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        gcp_path = row["gcp_path"]
        y = np.float32(row[CFG.label_col])

        x = physics_safe_normalise(load_raw_chw(gcp_path))
        x = torch.from_numpy(x)

        # Deterministic resize only. No crop, no flip, no rotation.
        if x.shape[-2:] != (CFG.image_size, CFG.image_size):
            x = F.interpolate(
                x.unsqueeze(0),
                size=(CFG.image_size, CFG.image_size),
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)

        return x, torch.tensor(y, dtype=torch.float32), gcp_path

train_ds = AIANPZPhysicsSafeDataset(train_df)
val_ds = AIANPZPhysicsSafeDataset(val_df)
test_ds = AIANPZPhysicsSafeDataset(test_df)

x0, y0, p0 = train_ds[0]
print("Sample:", x0.shape, x0.dtype, y0.item(), p0)
print("min/max/mean/std:", float(x0.min()), float(x0.max()), float(x0.mean()), float(x0.std()))


In [ ]:
# -----------------------------
# Data loaders and imbalance handling
# -----------------------------

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=CFG.num_workers > 0,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=CFG.num_workers > 0,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=CFG.num_workers > 0,
)

pos = float(train_df[CFG.label_col].sum())
neg = float(len(train_df) - pos)
pos_weight_value = neg / max(pos, 1.0)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

print("Train positives:", pos)
print("Train negatives:", neg)
print("pos_weight:", pos_weight_value)
print("Weighted sampler:", CFG.use_weighted_sampler)
print("Spatial augmentation:", CFG.spatial_augmentation)


In [ ]:
# -----------------------------
# AlexNet-v2 model
# -----------------------------

class AlexNetV2(nn.Module):
    def __init__(self, in_channels=6, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=11, stride=4, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.BatchNorm2d(384),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256 * 6 * 6, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x).squeeze(1)

model = AlexNetV2(in_channels=CFG.in_channels, dropout=CFG.dropout).to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print("Trainable parameters:", trainable_params)


In [ ]:
# -----------------------------
# Metrics helpers
# -----------------------------

def safe_auc(y_true, y_prob, kind="roc"):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob)) if kind == "roc" else float(average_precision_score(y_true, y_prob))

def confusion_at_threshold(y_true, y_prob, thr):
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_prob) >= thr).astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    tss = recall + specificity - 1.0

    total = tp + tn + fp + fn
    po = accuracy
    pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / (total * total) if total else 0.0
    hss = (po - pe) / (1 - pe) if abs(1 - pe) > 1e-12 else 0.0

    return {
        "threshold": float(thr),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "tss": float(tss),
        "hss": float(hss),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

def threshold_grid_metrics(y_true, y_prob):
    thresholds = np.arange(0.0, 1.0 + CFG.threshold_grid_step, CFG.threshold_grid_step)
    rows = [confusion_at_threshold(y_true, y_prob, thr) for thr in thresholds]
    grid = pd.DataFrame(rows)
    # Select by validation TSS. HSS then threshold are deterministic tie breakers.
    best = grid.sort_values(["tss", "hss", "threshold"], ascending=[False, False, True]).iloc[0].to_dict()
    return grid, best


In [ ]:
# -----------------------------
# Train / evaluate loops
# -----------------------------

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

def train_one_epoch(model, loader, epoch: int):
    model.train()
    total_loss, n = 0.0, 0
    t0 = time.time()

    for batch_idx, (xb, yb, _paths) in enumerate(loader, 1):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        bs = xb.size(0)
        total_loss += float(loss.detach().cpu()) * bs
        n += bs

        if batch_idx % 100 == 0 or batch_idx == len(loader):
            log_msg(
                f"epoch={epoch:02d} batch={batch_idx}/{len(loader)} "
                f"loss_running={total_loss/max(n,1):.4f} elapsed_min={(time.time()-t0)/60:.1f}"
            )

    return total_loss / max(n, 1)

@torch.no_grad()
def predict(model, loader, split_name: str):
    model.eval()
    y_true, y_prob, paths = [], [], []
    t0 = time.time()

    for batch_idx, (xb, yb, batch_paths) in enumerate(loader, 1):
        xb = xb.to(device, non_blocking=True)
        logits = model(xb)
        prob = torch.sigmoid(logits).detach().cpu().numpy()

        y_prob.extend(prob.tolist())
        y_true.extend(yb.numpy().astype(int).tolist())
        paths.extend(list(batch_paths))

        if batch_idx % 100 == 0 or batch_idx == len(loader):
            log_msg(
                f"predict split={split_name} batch={batch_idx}/{len(loader)} "
                f"elapsed_min={(time.time()-t0)/60:.1f}"
            )

    return pd.DataFrame({"gcp_path": paths, "y_true": y_true, "y_prob": y_prob})

def evaluate_split(model, loader, split_name: str):
    pred = predict(model, loader, split_name)
    y_true = pred["y_true"].values
    y_prob = pred["y_prob"].values
    grid, best = threshold_grid_metrics(y_true, y_prob)
    metrics = {
        "roc_auc": safe_auc(y_true, y_prob, "roc"),
        "pr_auc": safe_auc(y_true, y_prob, "pr"),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "at_0_5": confusion_at_threshold(y_true, y_prob, 0.5),
        "best_tss": best,
    }
    return metrics, grid, pred


In [ ]:
# -----------------------------
# Training with epoch-level artifacts
# -----------------------------

model_path = MODELS_DIR / f"{prefix}.pt"
history_path = METRICS_DIR / f"{prefix}_history.csv"
interim_path = METRICS_DIR / f"{prefix}_interim_metrics.json"

# Start a fresh progress log for this run.
progress_path.write_text("")
log_msg(f"Starting {prefix}")
log_msg(f"Progress log: {progress_path}")

history = []
best_val_tss = -999.0
best_epoch = None

for epoch in range(1, CFG.epochs + 1):
    epoch_start = time.time()
    train_loss = train_one_epoch(model, train_loader, epoch)
    val_metrics, _val_grid_tmp, _val_pred_tmp = evaluate_split(model, val_loader, split_name="val")
    val_best = val_metrics["best_tss"]
    val_tss = float(val_best["tss"])
    val_hss = float(val_best["hss"])
    lr_now = float(optimizer.param_groups[0]["lr"])

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_roc_auc": val_metrics["roc_auc"],
        "val_pr_auc": val_metrics["pr_auc"],
        "val_best_threshold": val_best["threshold"],
        "val_tss": val_tss,
        "val_hss": val_hss,
        "lr": lr_now,
        "epoch_elapsed_min": (time.time() - epoch_start) / 60,
    }
    history.append(row)

    # Save every epoch so an interrupted run is still diagnosable.
    pd.DataFrame(history).to_csv(history_path, index=False)
    interim_payload = {
        "experiment_name": CFG.experiment_name,
        "run_mode": CFG.run_mode,
        "fold_id": CFG.fold_id,
        "completed_epochs": epoch,
        "latest_epoch": row,
        "best_epoch_so_far": best_epoch,
        "best_val_tss_so_far": best_val_tss,
        "history": history,
    }
    interim_path.write_text(json.dumps(interim_payload, indent=2))

    log_msg(
        f"epoch={epoch:02d}/{CFG.epochs} COMPLETE "
        f"loss={train_loss:.4f} val_auc={val_metrics['roc_auc']:.4f} "
        f"val_pr={val_metrics['pr_auc']:.4f} val_thr={val_best['threshold']:.4f} "
        f"val_tss={val_tss:.4f} val_hss={val_hss:.4f} lr={lr_now:.2e} "
        f"epoch_min={(time.time()-epoch_start)/60:.1f}"
    )

    scheduler.step(val_tss)

    if val_tss > best_val_tss:
        best_val_tss = val_tss
        best_epoch = epoch
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "config": asdict(CFG),
                "best_epoch": best_epoch,
                "best_val_tss": best_val_tss,
                "trainable_parameters": trainable_params,
                "channel_stats": channel_stats_payload,
            },
            model_path,
        )
        log_msg(f"saved checkpoint -> {model_path}")

log_msg(f"Training complete. Best epoch={best_epoch}, best_val_tss={best_val_tss}")
log_msg(f"History saved: {history_path}")
log_msg(f"Interim metrics saved: {interim_path}")
log_msg(f"Model saved: {model_path}")


In [ ]:
# -----------------------------
# Final evaluation at best checkpoint
# -----------------------------

ckpt = torch.load(model_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

val_metrics, val_grid, val_pred = evaluate_split(model, val_loader, split_name="val_final")
test_metrics_raw, test_grid, test_pred = evaluate_split(model, test_loader, split_name="test_final")

selected_threshold = float(val_metrics["best_tss"]["threshold"])
test_at_selected = confusion_at_threshold(test_pred["y_true"].values, test_pred["y_prob"].values, selected_threshold)

test_metrics = {
    "roc_auc": test_metrics_raw["roc_auc"],
    "pr_auc": test_metrics_raw["pr_auc"],
    "brier_score": test_metrics_raw["brier_score"],
    "at_0_5": test_metrics_raw["at_0_5"],
    "best_tss": test_metrics_raw["best_tss"],
    "at_selected_threshold": test_at_selected,
}

val_pred_path = METRICS_DIR / f"{prefix}_val_predictions.csv"
test_pred_path = METRICS_DIR / f"{prefix}_test_predictions.csv"
val_grid_path = METRICS_DIR / f"{prefix}_val_threshold_grid.csv"
test_grid_path = METRICS_DIR / f"{prefix}_test_threshold_grid.csv"
metrics_path = METRICS_DIR / f"{prefix}_metrics.json"

val_pred.to_csv(val_pred_path, index=False)
test_pred.to_csv(test_pred_path, index=False)
val_grid.to_csv(val_grid_path, index=False)
test_grid.to_csv(test_grid_path, index=False)

metrics = {
    "experiment_name": CFG.experiment_name,
    "run_mode": CFG.run_mode,
    "fold_id": CFG.fold_id,
    "data_summary": data_summary,
    "physics_safety": {
        "label_source": CFG.label_col,
        "uses_embedded_npz_label": False,
        "spatial_augmentation": False,
        "random_flip": False,
        "random_rotation": False,
        "random_crop": False,
        "normalisation": channel_stats_payload["normalisation"],
        "normalisation_source": "training split only",
        "weighted_random_sampler": False,
        "threshold_protocol": "max TSS on validation applied unchanged to test",
    },
    "model": {
        "architecture": "AlexNet-v2 six-channel CNN with BatchNorm",
        "trainable_parameters": trainable_params,
        "image_size": CFG.image_size,
        "batch_size": CFG.batch_size,
        "epochs": CFG.epochs,
        "learning_rate": CFG.learning_rate,
        "weight_decay": CFG.weight_decay,
        "dropout": CFG.dropout,
        "pos_weight": pos_weight_value,
        "weighted_random_sampler": CFG.use_weighted_sampler,
    },
    "best_epoch": int(ckpt["best_epoch"]),
    "selected_threshold_from_validation": selected_threshold,
    "validation": val_metrics,
    "test": test_metrics,
    "artifacts": {
        "model_path": str(model_path.relative_to(ROOT)),
        "history_path": str(history_path.relative_to(ROOT)),
        "metrics_path": str(metrics_path.relative_to(ROOT)),
        "val_predictions": str(val_pred_path.relative_to(ROOT)),
        "test_predictions": str(test_pred_path.relative_to(ROOT)),
        "val_threshold_grid": str(val_grid_path.relative_to(ROOT)),
        "test_threshold_grid": str(test_grid_path.relative_to(ROOT)),
        "channel_stats": str(stats_path.relative_to(ROOT)),
        "progress_log": str(progress_path.relative_to(ROOT)),
    },
}

metrics_path.write_text(json.dumps(metrics, indent=2))
log_msg(f"Metrics saved: {metrics_path}")
metrics


In [ ]:
# -----------------------------
# Readable summary
# -----------------------------

test = metrics["test"]["at_selected_threshold"]
val_best = metrics["validation"]["best_tss"]

summary = f'''# AIA AlexNet-v2 Physics-Safe Fold 2015 Largecap Benchmark Summary

## Run

- Experiment: `{metrics["experiment_name"]}`
- Mode: `{metrics["run_mode"]}`
- Fold: `{metrics["fold_id"]}`
- Label: `{CFG.label_col}`
- Threshold rule: validation-selected max TSS, applied unchanged to test

## Physics-safe protocol

- Embedded NPZ label used: `False`
- Spatial augmentation used: `False`
- Random flip/rotation/crop used: `False`
- Normalisation: `{channel_stats_payload["normalisation"]}`
- Normalisation source: training split only
- Weighted random sampler: `False`
- Class imbalance handling: `BCEWithLogitsLoss(pos_weight=...)`

## Data

{pd.DataFrame(data_summary).to_markdown(index=False)}

## Model

- Architecture: `{metrics["model"]["architecture"]}`
- Trainable parameters: `{metrics["model"]["trainable_parameters"]}`
- Image size: `{CFG.image_size}`
- Batch size: `{CFG.batch_size}`
- Epochs: `{CFG.epochs}`
- Learning rate: `{CFG.learning_rate}`
- Weight decay: `{CFG.weight_decay}`
- Dropout: `{CFG.dropout}`
- Pos weight: `{metrics["model"]["pos_weight"]:.4f}`

## Best validation checkpoint

- Best epoch: `{metrics["best_epoch"]}`
- Validation-selected threshold: `{metrics["selected_threshold_from_validation"]:.4f}`
- Validation ROC-AUC: `{metrics["validation"]["roc_auc"]:.4f}`
- Validation PR-AUC: `{metrics["validation"]["pr_auc"]:.4f}`
- Validation best TSS: `{val_best["tss"]:.4f}`
- Validation best HSS: `{val_best["hss"]:.4f}`

## Test result at validation-selected threshold

- Test ROC-AUC: `{metrics["test"]["roc_auc"]:.4f}`
- Test PR-AUC: `{metrics["test"]["pr_auc"]:.4f}`
- Test Brier score: `{metrics["test"]["brier_score"]:.4f}`
- Test threshold: `{test["threshold"]:.4f}`
- Test Accuracy: `{test["accuracy"]:.4f}`
- Test Precision: `{test["precision"]:.4f}`
- Test Recall: `{test["recall"]:.4f}`
- Test Specificity: `{test["specificity"]:.4f}`
- Test F1: `{test["f1"]:.4f}`
- Test TSS: `{test["tss"]:.4f}`
- Test HSS: `{test["hss"]:.4f}`
- Confusion matrix: TP=`{test["tp"]}`, TN=`{test["tn"]}`, FP=`{test["fp"]}`, FN=`{test["fn"]}`

## Interpretation note

This notebook is intended as the official physics-safe AlexNet-v2 comparison point. It deliberately avoids non-physical image augmentations and uses train-derived channel statistics only, preserving chronological forecasting discipline.
'''

summary_path = METRICS_DIR / f"{prefix}_readable_summary.md"
summary_path.write_text(summary)
print(summary)
log_msg(f"Readable summary saved: {summary_path}")


In [ ]:
# -----------------------------
# Final artifact listing
# -----------------------------

for p in sorted(METRICS_DIR.glob(f"{prefix}_*")):
    print(p.relative_to(ROOT), p.stat().st_size)

print(model_path.relative_to(ROOT), model_path.stat().st_size)
print("Notebook 08B physics-safe complete.")
